# Quantum-Enhanced Multimodal Transformer Learning for Pipeline Predictive Maintenance

**Paper implementation:** DSMTPL + QE-CMFL + UC-PPMP  
**Runtime:** Google Colab / Python 3.10+  
**Important:** This notebook provides a runnable research implementation. Dataset schemas may vary; the loader attempts to map common columns and uses clearly marked synthetic data if downloads or schema matching fail. Synthetic outputs are not empirical validation.

## 1. Install dependencies

In [ ]:
!pip -q install numpy pandas scikit-learn matplotlib seaborn scipy shap pennylane

## 2. Imports and reproducibility

In [ ]:
import os, random, time, warnings, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score, mean_absolute_error,
    mean_squared_error, r2_score, roc_curve, precision_recall_curve)
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Dataset acquisition and profiling

Paper datasets:
- Pipeline thickness: [Hugging Face MustaphaL/pipeline-thickness-dataset](https://huggingface.co/datasets/MustaphaL/pipeline-thickness-dataset)
- Vibration leak detection: [Mendeley Data 575fy2tm7k](https://data.mendeley.com/datasets/575fy2tm7k/1)

Download these datasets manually if the sources require authentication or if their format changes. Place CSV files in `/content/data/`. The loader below searches that directory and supports common column names.

In [ ]:
DATA_DIR="/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
print("CSV files found:", [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")])

## 4. Load datasets (with transparent synthetic fallback)

In [ ]:
def read_csv_candidates(folder):
    frames=[]
    for fn in os.listdir(folder):
        if fn.lower().endswith(".csv"):
            try:
                df=pd.read_csv(os.path.join(folder,fn))
                if len(df): frames.append((fn,df))
            except Exception as e: print("Could not read",fn,":",e)
    return frames

files=read_csv_candidates(DATA_DIR)
for fn,df in files:
    print("\\n",fn,df.shape)
    display(df.head(3))

In [ ]:
def find_col(df, candidates):
    lower={str(c).strip().lower():c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower: return lower[cand.lower()]
    for c in df.columns:
        if any(x in str(c).lower() for x in candidates): return c
    return None

def make_synthetic_pipeline_data(n=5000, seed=SEED):
    rng=np.random.default_rng(seed)
    segments=np.array([f"SEG-{i:04d}" for i in range(max(50,n//50))])
    seg=rng.choice(segments,n)
    t=np.arange(n)
    initial=rng.normal(12.0,0.5,n)
    corrosion=np.abs(rng.normal(0.012,0.006,n))
    thickness=np.maximum(3.0, initial-corrosion*(t%100)+rng.normal(0,.08,n))
    vibration=rng.normal(0,1,n)+0.18*(12-thickness)+rng.normal(0,.25,n)
    leak_prob=1/(1+np.exp(-(-3.2+1.2*np.abs(vibration)+.65*(12-thickness))))
    leak=(rng.random(n)<leak_prob).astype(int)
    dates=pd.date_range("2018-01-01",periods=n,freq="D")
    return pd.DataFrame({"timestamp":dates,"segment_id":seg,"wall_thickness":thickness,
                        "corrosion_rate":corrosion,"vibration_rms":np.abs(vibration),
                        "vibration_mean":vibration,"pressure":rng.normal(6,0.5,n),
                        "temperature":rng.normal(45,5,n),"leakage":leak})

thickness_df=None; vibration_df=None
for fn,df in files:
    cols=[str(c).lower() for c in df.columns]
    if any(("thick" in c or "wall" in c) for c in cols) and thickness_df is None: thickness_df=df.copy()
    if any(("vib" in c or "leak" in c or "lda" in c) for c in cols) and vibration_df is None: vibration_df=df.copy()

if thickness_df is None and vibration_df is None:
    print("No compatible source CSV detected. Using marked synthetic demonstration data.")
    base=make_synthetic_pipeline_data()
    thickness_df=base.copy(); vibration_df=base.copy()
elif thickness_df is None:
    thickness_df=make_synthetic_pipeline_data()
elif vibration_df is None:
    vibration_df=thickness_df.copy()

print("Thickness source shape:", thickness_df.shape)
print("Vibration source shape:", vibration_df.shape)

## 5. Normalize source schemas and create temporal windows

In [ ]:
def normalize_structural(df):
    d=df.copy()
    thick=find_col(d,["wall_thickness","thickness","pipe_thickness","remaining_thickness"])
    if thick is None:
        nums=d.select_dtypes(include=np.number).columns
        if len(nums)==0: raise ValueError("No numeric structural feature columns found.")
        thick=nums[0]
        print("WARNING: thickness target inferred as",thick)
    timecol=find_col(d,["timestamp","date","datetime","time"])
    segcol=find_col(d,["segment_id","segment","pipeline_id","pipe_id","asset_id"])
    if timecol: d["timestamp"]=pd.to_datetime(d[timecol],errors="coerce")
    else: d["timestamp"]=pd.date_range("2000-01-01",periods=len(d),freq="D")
    d["segment_id"]=d[segcol].astype(str) if segcol else "SEG-0"
    d["wall_thickness"]=pd.to_numeric(d[thick],errors="coerce")
    d=d.dropna(subset=["wall_thickness"]).sort_values(["segment_id","timestamp"])
    return d

def normalize_vibration(df):
    d=df.copy()
    leak=find_col(d,["leakage","leak","label","target","class","fault"])
    if leak is None:
        d["leakage"]=0
        print("WARNING: no leakage label found; placeholder labels set to 0 (not valid evaluation).")
    else:
        raw=d[leak]
        if raw.dtype==object:
            d["leakage"]=raw.astype(str).str.lower().str.contains("leak|fault|1|yes|true").astype(int)
        else:
            d["leakage"]=(pd.to_numeric(raw,errors="coerce").fillna(0)>0).astype(int)
    d["segment_id"]=d[find_col(d,["segment_id","segment","pipeline_id","pipe_id","asset_id"])].astype(str) if find_col(d,["segment_id","segment","pipeline_id","pipe_id","asset_id"]) else "SEG-0"
    timecol=find_col(d,["timestamp","date","datetime","time"])
    d["timestamp"]=pd.to_datetime(d[timecol],errors="coerce") if timecol else pd.date_range("2000-01-01",periods=len(d),freq="D")
    return d.sort_values(["segment_id","timestamp"])

struct=normalize_structural(thickness_df)
vib=normalize_vibration(vibration_df)
print(struct.columns.tolist())
print(vib.columns.tolist())
display(struct.head())

In [ ]:
def numeric_feature_columns(df, exclude):
    return [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]

struct_feats=numeric_feature_columns(struct,["wall_thickness"])
vib_feats=numeric_feature_columns(vib,["leakage"])
if not struct_feats: struct_feats=["wall_thickness"]; struct["wall_thickness"]=struct["wall_thickness"]
if not vib_feats:
    vib["vibration_proxy"]=0.0; vib_feats=["vibration_proxy"]
print("Structural features:",struct_feats)
print("Vibration features:",vib_feats)

## 6. Chronological split, imputation, scaling, and sequence building

In [ ]:
# Split by chronological index; scaler/imputer fit only on train portion.
def temporal_split(df, train=.70, val=.15):
    df=df.sort_values("timestamp").reset_index(drop=True)
    n=len(df); a=int(n*train); b=int(n*(train+val))
    return df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()

s_train,s_val,s_test=temporal_split(struct)
v_train,v_val,v_test=temporal_split(vib)

s_imputer=SimpleImputer(strategy="median")
s_scaler=StandardScaler()
Xst=s_imputer.fit_transform(s_train[struct_feats]); Xst=s_scaler.fit_transform(Xst)
Xsv=s_scaler.transform(s_imputer.transform(s_val[struct_feats]))
Xse=s_scaler.transform(s_imputer.transform(s_test[struct_feats]))

v_imputer=SimpleImputer(strategy="median")
v_scaler=StandardScaler()
Xvt=v_imputer.fit_transform(v_train[vib_feats]); Xvt=v_scaler.fit_transform(Xvt)
Xvv=v_scaler.transform(v_imputer.transform(v_val[vib_feats]))
Xve=v_scaler.transform(v_imputer.transform(v_test[vib_feats]))

def windows(X,y,seq_len=16):
    xs=[]; ys=[]
    if len(X)<=seq_len: seq_len=max(1,len(X)-1)
    for i in range(seq_len,len(X)):
        xs.append(X[i-seq_len:i]); ys.append(y[i])
    if not xs: return np.zeros((0,seq_len,X.shape[1]),np.float32),np.array([])
    return np.asarray(xs,dtype=np.float32),np.asarray(ys)

SEQ_LEN=16
ys_train=s_train["wall_thickness"].to_numpy()
ys_val=s_val["wall_thickness"].to_numpy()
ys_test=s_test["wall_thickness"].to_numpy()
Xs_train,yt_train=windows(Xst,ys_train,SEQ_LEN)
Xs_val,yt_val=windows(Xsv,ys_val,SEQ_LEN)
Xs_test,yt_test=windows(Xse,ys_test,SEQ_LEN)

yv_train=v_train["leakage"].to_numpy()
yv_val=v_val["leakage"].to_numpy()
yv_test=v_test["leakage"].to_numpy()
Xv_train,yv_train=windows(Xvt,yv_train,SEQ_LEN)
Xv_val,yv_val=windows(Xvv,yv_val,SEQ_LEN)
Xv_test,yv_test=windows(Xve,yv_test,SEQ_LEN)

# Paired demo streams by chronological sequence length. If source records are unrelated,
# this alignment is only a demonstration and is not true sensor-level pairing.
def align_streams(A,B):
    n=min(len(A),len(B))
    return A[:n],B[:n]
Xs_train,Xv_train=align_streams(Xs_train,Xv_train)
Xs_val,Xv_val=align_streams(Xs_val,Xv_val)
Xs_test,Xv_test=align_streams(Xs_test,Xv_test)
print("Train:",Xs_train.shape,Xv_train.shape)

## 7. DSMTPL: dual-stream Transformer, prototype attention, reliability-gated fusion

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=512):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len).float().unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class DSMTPL(nn.Module):
    def __init__(self,ns,nv,d=64,heads=4,layers=2):
        super().__init__()
        self.s_proj=nn.Linear(ns,d); self.v_proj=nn.Linear(nv,d)
        self.pos=PositionalEncoding(d)
        enc=lambda: nn.TransformerEncoder(nn.TransformerEncoderLayer(d,heads,4*d,batch_first=True,dropout=.1),layers)
        self.s_enc=enc(); self.v_enc=enc()
        self.s_proto=nn.Parameter(torch.randn(4,d)*.1)
        self.v_proto=nn.Parameter(torch.randn(4,d)*.1)
        self.s_to_v=nn.MultiheadAttention(d,heads,batch_first=True)
        self.v_to_s=nn.MultiheadAttention(d,heads,batch_first=True)
        self.reliability=nn.Sequential(nn.Linear(2*d, d//2),nn.ReLU(),nn.Linear(d//2,2))
        self.fuse=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.LayerNorm(d))
    def forward(self,s,v):
        s=self.s_enc(self.pos(self.s_proj(s)))
        v=self.v_enc(self.pos(self.v_proj(v)))
        # Cross-modal reference using learned modality prototypes.
        sp=self.s_proto.unsqueeze(0).expand(s.size(0),-1,-1)
        vp=self.v_proto.unsqueeze(0).expand(v.size(0),-1,-1)
        sctx,_=self.s_to_v(s,vp)
        vctx,_=self.v_to_s(v,sp)
        spooled=s.mean(1); vpooled=v.mean(1)
        logits=self.reliability(torch.cat([spooled,vpooled],-1))
        weights=torch.softmax(logits,dim=-1)
        sf=(s+sctx).mean(1)*weights[:,0:1]
        vf=(v+vctx).mean(1)*weights[:,1:2]
        h=self.fuse(torch.cat([sf,vf],-1))
        return h,weights,s.mean(1),v.mean(1)

class MultiTaskModel(nn.Module):
    def __init__(self,ns,nv):
        super().__init__()
        self.dsm=DSMTPL(ns,nv)
        self.leak=nn.Linear(64,1)
        self.thickness=nn.Linear(64,1)
        self.rul=nn.Linear(64,3)
    def forward(self,s,v):
        h,w,hs,hv=self.dsm(s,v)
        return {"health":h,"leak_logit":self.leak(h).squeeze(-1),
                "thickness":self.thickness(h).squeeze(-1),
                "rul_quantiles":self.rul(h),"reliability":w,
                "struct_repr":hs,"vib_repr":hv}

model=MultiTaskModel(Xs_train.shape[-1],Xv_train.shape[-1]).to(DEVICE)
print(model)

## 8. QE-CMFL: lightweight differentiable quantum-inspired circuit (simulator)

This notebook implements a differentiable quantum-inspired circuit layer using PyTorch operations (RY-like angle encoding, trainable rotations, entanglement-inspired interactions, and X/Z/ZZ readouts). It is runnable without a quantum hardware backend. Circuit depth/qubit count are reported as architecture settings, not hardware execution results. For actual quantum execution, replace this layer with PennyLane/Qiskit backend integration.

In [ ]:
class QuantumEnhancedLayer(nn.Module):
    def __init__(self,d=64,n_qubits=8,depth=2):
        super().__init__()
        self.n_qubits=n_qubits; self.depth=depth
        self.proj=nn.Linear(d,n_qubits)
        self.theta=nn.Parameter(torch.randn(depth,n_qubits,3)*.1)
        self.readout_dim=3*n_qubits+(n_qubits-1)
        self.out=nn.Linear(self.readout_dim,d)
    def forward(self,h,reliability):
        x=torch.tanh(self.proj(h))*math.pi
        # Reliability modulation, one weight per stream, averaged to a gate.
        rel=reliability.mean(dim=1,keepdim=True)
        angles=x*rel
        z=torch.sin(angles+self.theta[0,:,0])
        xx=torch.cos(angles+self.theta[0,:,1])
        yy=torch.sin(2*angles+self.theta[0,:,2])
        for layer in range(self.depth):
            # differentiable nearest-neighbor entanglement proxy
            z=z+0.25*torch.roll(z,1,dims=1)*torch.cos(self.theta[layer,:,0])
            xx=xx*torch.cos(self.theta[layer,:,1])-yy*torch.sin(self.theta[layer,:,1])
            yy=xx*torch.sin(self.theta[layer,:,2])+yy*torch.cos(self.theta[layer,:,2])
        zz=z[:,:-1]*z[:,1:]
        q=torch.cat([xx,yy,z,zz],dim=1)
        return self.out(q)

class QuantumMultiTask(nn.Module):
    def __init__(self,base):
        super().__init__()
        self.base=base
        self.q=QuantumEnhancedLayer(64,8,2)
        self.gate=nn.Sequential(nn.Linear(128,64),nn.Sigmoid())
        self.leak=nn.Linear(64,1); self.thick=nn.Linear(64,1); self.rul=nn.Linear(64,3)
    def forward(self,s,v):
        o=self.base(s,v)
        q=self.q(o["health"],o["reliability"])
        g=self.gate(torch.cat([o["health"],q],dim=-1))
        h=g*q+(1-g)*o["health"]
        o.update(health_q=h,leak_logit=self.leak(h).squeeze(-1),
                thickness=self.thick(h).squeeze(-1),rul_quantiles=self.rul(h))
        return o

qmodel=QuantumMultiTask(model).to(DEVICE)

## 9. Training utilities and multitask objective

In [ ]:
class MultiDataset(Dataset):
    def __init__(self,s,v,leak,thick):
        n=min(len(s),len(v),len(leak),len(thick))
        self.s=torch.tensor(s[:n],dtype=torch.float32)
        self.v=torch.tensor(v[:n],dtype=torch.float32)
        self.leak=torch.tensor(leak[:n],dtype=torch.float32)
        self.thick=torch.tensor(thick[:n],dtype=torch.float32)
    def __len__(self): return len(self.s)
    def __getitem__(self,i): return self.s[i],self.v[i],self.leak[i],self.thick[i]

def make_rul(thickness, retirement=6.0, rate=.01):
    return np.maximum(0,(thickness-retirement)/max(rate,1e-6))

# Align labels and streams for demo
n=min(len(Xs_train),len(Xv_train),len(yv_train),len(yt_train))
train_ds=MultiDataset(Xs_train[:n],Xv_train[:n],yv_train[:n],yt_train[:n])
n=min(len(Xs_val),len(Xv_val),len(yv_val),len(yt_val))
val_ds=MultiDataset(Xs_val[:n],Xv_val[:n],yv_val[:n],yt_val[:n])
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=128,shuffle=False)

def pinball(pred,target,q):
    err=target-pred
    return torch.maximum(q*err,(q-1)*err).mean()

def run_epoch(net,loader,opt=None):
    train=opt is not None
    net.train(train); total=0; count=0
    bce=nn.BCEWithLogitsLoss(); mse=nn.MSELoss()
    for s,v,l,t in loader:
        s,v,l,t=s.to(DEVICE),v.to(DEVICE),l.to(DEVICE),t.to(DEVICE)
        if train: opt.zero_grad()
        out=net(s,v)
        loss_cls=bce(out["leak_logit"],l)
        loss_reg=mse(out["thickness"],t)
        # quantile targets derived from thickness and illustrative corrosion rate
        rul=torch.clamp((t-6.0)/.01,min=0)
        qs=out["rul_quantiles"]
        loss_q=pinball(qs[:,0],rul,.1)+pinball(qs[:,1],rul,.5)+pinball(qs[:,2],rul,.9)
        loss=loss_cls+loss_reg+.15*loss_q
        if train: loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(),1.0); opt.step()
        total+=loss.item()*len(s); count+=len(s)
    return total/max(count,1)

EPOCHS=8
optimizer=torch.optim.AdamW(qmodel.parameters(),lr=1e-3,weight_decay=1e-4)
history={"train":[],"val":[]}
start=time.time()
for epoch in range(EPOCHS):
    tr=run_epoch(qmodel,train_loader,optimizer)
    va=run_epoch(qmodel,val_loader)
    history["train"].append(tr); history["val"].append(va)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} train={tr:.4f} val={va:.4f}")
print("Training seconds:",round(time.time()-start,2))

## 10. Training curves

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["train"],label="Train")
plt.plot(history["val"],label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Combined loss"); plt.title("DSMTPL + QE-CMFL convergence")
plt.legend(); plt.grid(True); plt.show()

## 11. Test evaluation: leakage classification, thickness, RUL quantiles

In [ ]:
test_ds=MultiDataset(Xs_test,Xv_test,yv_test,yt_test)
test_loader=DataLoader(test_ds,batch_size=128,shuffle=False)
qmodel.eval()
preds={"leak":[],"thick":[],"rul":[],"true_leak":[],"true_thick":[]}
with torch.no_grad():
    for s,v,l,t in test_loader:
        out=qmodel(s.to(DEVICE),v.to(DEVICE))
        preds["leak"].extend(torch.sigmoid(out["leak_logit"]).cpu().numpy())
        preds["thick"].extend(out["thickness"].cpu().numpy())
        preds["rul"].extend(out["rul_quantiles"].cpu().numpy())
        preds["true_leak"].extend(l.numpy()); preds["true_thick"].extend(t.numpy())
for k in preds: preds[k]=np.asarray(preds[k])

In [ ]:
y_true=preds["true_leak"].astype(int)
prob=preds["leak"]
y_hat=(prob>=.5).astype(int)
def safe_auc(y,p):
    return roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
def safe_ap(y,p):
    return average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
cm=confusion_matrix(y_true,y_hat,labels=[0,1])
tn,fp,fn,tp=cm.ravel()
metrics_cls={
 "Accuracy":accuracy_score(y_true,y_hat),
 "Precision":precision_score(y_true,y_hat,zero_division=0),
 "Recall":recall_score(y_true,y_hat,zero_division=0),
 "Specificity":tn/(tn+fp) if tn+fp else 0,
 "F1":f1_score(y_true,y_hat,zero_division=0),
 "ROC-AUC":safe_auc(y_true,prob),
 "PR-AUC":safe_ap(y_true,prob)}
yt=preds["true_thick"]; yp=preds["thick"]
metrics_reg={"MAE":mean_absolute_error(yt,yp),"RMSE":mean_squared_error(yt,yp)**.5,
             "R2":r2_score(yt,yp),"MAPE(%)":np.mean(np.abs((yt-yp)/np.maximum(np.abs(yt),1e-6)))*100}
rul_true=np.maximum(0,(yt-6.0)/.01)
rq=preds["rul"]
rul_med=rq[:,1]
metrics_rul={"RUL-MAE":mean_absolute_error(rul_true,rul_med),
              "RUL-RMSE":mean_squared_error(rul_true,rul_med)**.5,
              "RUL-R2":r2_score(rul_true,rul_med)}
lower=np.minimum(rq[:,0],rq[:,2]); upper=np.maximum(rq[:,0],rq[:,2])
metrics_unc={"PICP(%)":np.mean((rul_true>=lower)&(rul_true<=upper))*100,
             "MPIW":np.mean(upper-lower)}
print("Leakage metrics"); display(pd.Series(metrics_cls).to_frame("Value"))
print("Thickness metrics"); display(pd.Series(metrics_reg).to_frame("Value"))
print("RUL metrics"); display(pd.Series(metrics_rul).to_frame("Value"))
print("Uncertainty metrics"); display(pd.Series(metrics_unc).to_frame("Value"))

## 12. Confusion matrix, ROC and precision-recall curves

In [ ]:
fig,ax=plt.subplots(figsize=(5,4)); ax.imshow(cm)
ax.set_xticks([0,1],["Non-leak","Leak"]); ax.set_yticks([0,1],["Non-leak","Leak"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Leakage confusion matrix")
for i in range(2):
 for j in range(2): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.show()

if len(np.unique(y_true))>1:
 fpr,tpr,_=roc_curve(y_true,prob)
 plt.figure(figsize=(5,4)); plt.plot(fpr,tpr,label=f"AUC={metrics_cls['ROC-AUC']:.3f}")
 plt.plot([0,1],[0,1],"--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curve"); plt.legend(); plt.grid(); plt.show()
 p,r,_=precision_recall_curve(y_true,prob)
 plt.figure(figsize=(5,4)); plt.plot(r,p,label=f"AP={metrics_cls['PR-AUC']:.3f}")
 plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-recall curve"); plt.legend(); plt.grid(); plt.show()

## 13. Actual vs predicted thickness and RUL quantiles

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(yt[:250],label="Actual thickness")
plt.plot(yp[:250],label="Predicted thickness")
plt.xlabel("Test sample"); plt.ylabel("Wall thickness"); plt.title("Actual vs predicted thickness")
plt.legend(); plt.grid(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(rul_true[:250],label="Actual derived RUL")
plt.plot(rq[:250,1],label="Median predicted RUL")
plt.fill_between(np.arange(min(250,len(rq))),lower[:250],upper[:250],alpha=.2,label="Predicted interval")
plt.xlabel("Test sample"); plt.ylabel("RUL"); plt.title("RUL prognosis")
plt.legend(); plt.grid(); plt.show()

## 14. Baseline models: 1D CNN, XGBoost (optional), LSTM, CNN-LSTM

In [ ]:
# Flatten temporal windows for classical baseline and use sequence tensors for deep baselines.
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import classification_report
def flatten_pair(s,v): return np.concatenate([s.reshape(len(s),-1),v.reshape(len(v),-1)],axis=1)
Xtr=flatten_pair(Xs_train,Xv_train); Xte=flatten_pair(Xs_test,Xv_test)
ntr=min(len(Xtr),len(yv_train)); nte=min(len(Xte),len(yv_test))
baseline_results={}
if ntr>0 and nte>0:
    clf=HistGradientBoostingClassifier(max_iter=100,random_state=SEED)
    clf.fit(Xtr[:ntr],yv_train[:ntr])
    bp=clf.predict_proba(Xte[:nte])[:,1] if len(np.unique(yv_train[:ntr]))>1 else np.zeros(nte)
    baseline_results["HistGradientBoosting"]={"F1":f1_score(yv_test[:nte],bp>=.5,zero_division=0),
                                               "ROC-AUC":safe_auc(yv_test[:nte],bp)}
    reg=HistGradientBoostingRegressor(max_iter=100,random_state=SEED)
    reg.fit(Xtr[:min(ntr,len(yt_train))],yt_train[:min(ntr,len(yt_train))])
    breg=reg.predict(Xte[:min(nte,len(yt_test))])
    baseline_results["HistGradientBoosting-regression"]={"MAE":mean_absolute_error(yt_test[:len(breg)],breg),
                                                          "RMSE":mean_squared_error(yt_test[:len(breg)],breg)**.5}
display(pd.DataFrame(baseline_results).T)

In [ ]:
class SeqBaseline(nn.Module):
    def __init__(self,ns,nv,kind="lstm"):
        super().__init__()
        self.kind=kind
        self.conv_s=nn.Conv1d(ns,32,3,padding=1); self.conv_v=nn.Conv1d(nv,32,3,padding=1)
        if kind=="lstm":
            self.rnn=nn.LSTM(64,64,batch_first=True)
        elif kind=="cnn_lstm":
            self.conv=nn.Conv1d(64,64,3,padding=1); self.rnn=nn.LSTM(64,64,batch_first=True)
        self.cls=nn.Linear(64,1); self.reg=nn.Linear(64,1)
    def forward(self,s,v):
        a=torch.relu(self.conv_s(s.transpose(1,2))).transpose(1,2)
        b=torch.relu(self.conv_v(v.transpose(1,2))).transpose(1,2)
        x=torch.cat([a,b],-1)
        if self.kind=="cnn_lstm": x=torch.relu(self.conv(x.transpose(1,2))).transpose(1,2)
        x,_=self.rnn(x); h=x[:,-1]
        return self.cls(h).squeeze(-1),self.reg(h).squeeze(-1)

def train_seq_baseline(kind,epochs=3):
    net=SeqBaseline(Xs_train.shape[-1],Xv_train.shape[-1],kind).to(DEVICE)
    opt=torch.optim.Adam(net.parameters(),lr=1e-3)
    for ep in range(epochs):
        net.train()
        for s,v,l,t in train_loader:
            s,v,l,t=s.to(DEVICE),v.to(DEVICE),l.to(DEVICE),t.to(DEVICE)
            opt.zero_grad(); lc,lr=net(s,v)
            loss=nn.functional.binary_cross_entropy_with_logits(lc,l)+nn.functional.mse_loss(lr,t)
            loss.backward(); opt.step()
    net.eval(); pp=[]; rr=[]
    with torch.no_grad():
        for s,v,l,t in test_loader:
            lc,lr=net(s.to(DEVICE),v.to(DEVICE)); pp.extend(torch.sigmoid(lc).cpu().numpy()); rr.extend(lr.cpu().numpy())
    return net,np.array(pp),np.array(rr)

deep_baselines={}
for kind in ["lstm","cnn_lstm"]:
    try:
        net,pp,rr=train_seq_baseline(kind,epochs=3)
        deep_baselines[kind]={"F1":f1_score(y_true[:len(pp)],pp>=.5,zero_division=0),
                               "MAE":mean_absolute_error(yt[:len(rr)],rr)}
    except Exception as e: print(kind,"baseline skipped:",e)
display(pd.DataFrame(deep_baselines).T)
print("1D CNN is represented by the convolutional stream; add a standalone Conv1D head if desired.")

## 15. SHAP / feature attribution and modality contribution

In [ ]:
# Lightweight permutation importance, applicable even when SHAP isn't installed.
from sklearn.inspection import permutation_importance
try:
    if 'clf' in globals():
        # names are flattened temporal features; aggregate importance by modality
        perm=permutation_importance(clf,Xte[:nte],yv_test[:nte],n_repeats=3,random_state=SEED)
        imp=perm.importances_mean
        ns=len(struct_feats)*SEQ_LEN; nv=len(vib_feats)*SEQ_LEN
        structural_imp=float(np.sum(imp[:ns])); vibration_imp=float(np.sum(imp[ns:ns+nv]))
        display(pd.Series({"Structural stream":structural_imp,"Vibration stream":vibration_imp}).to_frame("Permutation importance"))
        top=pd.Series(imp).sort_values(ascending=False).head(20)
        top.sort_values().plot(kind="barh",figsize=(7,5),title="Top flattened-feature permutation importance")
        plt.xlabel("Importance"); plt.show()
except Exception as e: print("Permutation importance unavailable:",e)

try:
    import shap
    print("SHAP installed. For larger datasets, run SHAP on a small background/sample to reduce runtime.")
    if 'clf' in globals():
        explainer=shap.Explainer(clf.predict, Xtr[:min(100,len(Xtr))])
        sv=explainer(Xte[:min(100,len(Xte))])
        shap.plots.beeswarm(sv,max_display=15)
except Exception as e: print("SHAP analysis skipped:",e)

## 16. Maintenance priority ranking, Recall@K, NDCG@K

In [ ]:
# Risk combines predicted leakage probability, low median RUL, and wide uncertainty.
rul_width=np.maximum(upper-lower,0)
rul_norm=rul_med/(np.max(rul_med)+1e-8)
unc_norm=rul_width/(np.max(rul_width)+1e-8)
risk=0.60*prob+0.30*(1-rul_norm)+0.10*unc_norm
rank_df=pd.DataFrame({"sample_id":np.arange(len(risk)),"leak_probability":prob,
                      "rul_median":rul_med,"rul_lower":lower,"rul_upper":upper,
                      "risk_score":risk,"actual_leak":y_true})
rank_df=rank_df.sort_values("risk_score",ascending=False).reset_index(drop=True)
display(rank_df.head(15))

def recall_at_k(y,score,k):
    k=min(k,len(y)); positives=np.sum(y)
    if positives==0:return np.nan
    idx=np.argsort(score)[::-1][:k]
    return np.sum(y[idx])/positives
def ndcg_at_k(y,score,k):
    k=min(k,len(y)); idx=np.argsort(score)[::-1][:k]
    rel=np.asarray(y)[idx]
    dcg=np.sum((2**rel-1)/np.log2(np.arange(2,k+2)))
    ideal=np.sort(y)[::-1][:k]
    idcg=np.sum((2**ideal-1)/np.log2(np.arange(2,k+2)))
    return dcg/idcg if idcg else 0
for k in [10,20,50]:
    print(f"K={k}: Recall@K={recall_at_k(y_true,risk,k):.4f}, NDCG@K={ndcg_at_k(y_true,risk,k):.4f}")

## 17. Ablation study (progressive components)

In [ ]:
# Train/evaluate a compact ablation subset. Results are measured on this run; no fabricated scores.
# Components: classical fused features, DSMTPL, and DSMTPL+quantum.
def eval_leak(net,loader):
    net.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for s,v,l,t in loader:
            out=net(s.to(DEVICE),v.to(DEVICE))
            ys.extend(l.numpy()); ps.extend(torch.sigmoid(out["leak_logit"]).cpu().numpy())
    ys=np.asarray(ys); ps=np.asarray(ps)
    return {"Accuracy":accuracy_score(ys,ps>=.5),"Precision":precision_score(ys,ps>=.5,zero_division=0),
            "Recall":recall_score(ys,ps>=.5,zero_division=0),"F1":f1_score(ys,ps>=.5,zero_division=0),
            "ROC-AUC":safe_auc(ys,ps)}

# The trained full model is the quantum-enhanced row. DSMTPL-only is a separate model.
dsm_only=MultiTaskModel(Xs_train.shape[-1],Xv_train.shape[-1]).to(DEVICE)
opt_dsm=torch.optim.AdamW(dsm_only.parameters(),lr=1e-3)
for ep in range(3): run_epoch(dsm_only,train_loader,opt_dsm)
ablation={"DSMTPL only":eval_leak(dsm_only,test_loader),
          "DSMTPL + QE-CMFL":eval_leak(qmodel,test_loader)}
display(pd.DataFrame(ablation).T)

## 18. Robustness analysis

In [ ]:
# Evaluate leakage performance after Gaussian perturbation to standardized inputs.
def perturb_eval(std):
    noisy_s=Xs_test+np.random.normal(0,std,Xs_test.shape).astype(np.float32)
    noisy_v=Xv_test+np.random.normal(0,std,Xv_test.shape).astype(np.float32)
    ds=MultiDataset(noisy_s,noisy_v,yv_test,yt_test)
    return eval_leak(qmodel,DataLoader(ds,batch_size=128))
robust={}
for noise in [0.0,0.05,0.10,0.20]:
    robust[f"noise_{noise}"]=perturb_eval(noise)
display(pd.DataFrame(robust).T)

## 19. Statistical analysis / ANOVA

In [ ]:
# Example paired comparison across per-sample binary correctness for two models.
# McNemar's test is more suitable for paired classification correctness than one-way ANOVA.
try:
    from statsmodels.stats.contingency_tables import mcnemar
    def correctness(net,loader):
        net.eval(); y=[]; p=[]
        with torch.no_grad():
            for s,v,l,t in loader:
                o=net(s.to(DEVICE),v.to(DEVICE))
                y.extend(l.numpy()); p.extend((torch.sigmoid(o["leak_logit"]).cpu().numpy()>=.5).astype(int))
        return np.asarray(y),np.asarray(p)
    y1,p1=correctness(dsm_only,test_loader); y2,p2=correctness(qmodel,test_loader)
    table=np.array([[np.sum((p1==y1)&(p2==y2)),np.sum((p1==y1)&(p2!=y2))],
                    [np.sum((p1!=y1)&(p2==y2)),np.sum((p1!=y1)&(p2!=y2))]])
    print("Paired correctness table:",table)
    print("McNemar:",mcnemar(table,exact=False,correction=True))
except Exception as e: print("Statistical test skipped:",e)

# One-way ANOVA can compare repeated-run scores, but requires multiple independent seeds.
# This cell demonstrates the valid form; rerun training with distinct seeds before reporting p-values.
from scipy.stats import f_oneway
print("ANOVA template: supply arrays of independent run-level scores for each model.")

## 20. Quantum/computational efficiency report

In [ ]:
efficiency=pd.DataFrame([
 {"Model":"DSMTPL","Qubits":0,"Circuit depth":0,"Note":"Classical Transformer"},
 {"Model":"QE-CMFL","Qubits":8,"Circuit depth":2,"Note":"Differentiable simulator proxy; not hardware execution"}
])
display(efficiency)
print("Quantum execution time/shots: unavailable for this simulator proxy.")
print("Training wall time recorded above; inference latency can be measured below.")
qmodel.eval()
sample_s=torch.tensor(Xs_test[:min(128,len(Xs_test))],dtype=torch.float32,device=DEVICE)
sample_v=torch.tensor(Xv_test[:min(128,len(Xv_test))],dtype=torch.float32,device=DEVICE)
if len(sample_s):
    for _ in range(2):
        with torch.no_grad(): qmodel(sample_s,sample_v)
    t0=time.perf_counter()
    with torch.no_grad(): qmodel(sample_s,sample_v)
    print("Batch inference latency (seconds):",time.perf_counter()-t0)
    print("Batch size:",len(sample_s))

## 21. Export metrics, predictions, and maintenance ranking

In [ ]:
all_metrics={**{f"Leakage/{k}":v for k,v in metrics_cls.items()},
             **{f"Thickness/{k}":v for k,v in metrics_reg.items()},
             **{f"RUL/{k}":v for k,v in metrics_rul.items()},
             **{f"Uncertainty/{k}":v for k,v in metrics_unc.items()}}
metrics_df=pd.DataFrame({"Metric":list(all_metrics.keys()),"Value":list(all_metrics.values())})
pred_df=pd.DataFrame({"actual_leak":y_true,"leak_probability":prob,"predicted_leak":y_hat,
                      "actual_thickness":yt,"predicted_thickness":yp,
                      "rul_q10":rq[:,0],"rul_q50":rq[:,1],"rul_q90":rq[:,2]})
OUT="/content/pipeline_maintenance_results"
os.makedirs(OUT,exist_ok=True)
metrics_df.to_csv(os.path.join(OUT,"metrics.csv"),index=False)
pred_df.to_csv(os.path.join(OUT,"test_predictions.csv"),index=False)
rank_df.to_csv(os.path.join(OUT,"maintenance_priority.csv"),index=False)
efficiency.to_csv(os.path.join(OUT,"quantum_efficiency.csv"),index=False)
print("Saved files:")
print(os.listdir(OUT))

## 22. Download results (Colab)

In [ ]:
# Run in Google Colab to download a ZIP of generated CSV files.
import shutil
zip_path=shutil.make_archive("/content/pipeline_maintenance_results","zip",OUT)
print("ZIP:",zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab; ZIP is saved at",zip_path)

## Notes for research reporting

- Confirm the dataset target columns, unit conventions, segment IDs, timestamps, and leakage labels before publication.
- Thickness and vibration sources may not contain paired observations. This notebook aligns windows by order for a runnable demonstration; scientifically valid multimodal fusion requires verified temporal/segment correspondence.
- The RUL labels are derived from a configurable retirement-thickness threshold and assumed corrosion rate when ground-truth RUL is unavailable. Replace this with measured maintenance/retirement histories.
- The simulator-based quantum layer is not a quantum hardware result. Report real qubit execution, shots, circuit depth, and runtime only after using a quantum backend.
- Ablation, robustness, and statistical comparisons should be repeated across multiple seeds and/or independent assets. Do not present one synthetic run as evidence of generalization.